# Day 1 — Document Ingestion
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 1 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

This notebook walks through the full Day 1 pipeline step by step: parsing a real clinical
guideline PDF, choosing a chunking strategy, generating embeddings, and building a
queryable vector index — the same steps implemented in `ingest.py`, but broken apart here
so you can inspect what happens at each stage before you rely on the script.

**By the end of this notebook you will be able to:**
1. Explain why grounding — not raw model memory — matters in clinical AI
2. Parse a PDF and inspect its extracted structure
3. Compare fixed-size vs. section-aware chunking on the same document
4. Generate an embedding and explain what the resulting vector represents
5. Build a persisted vector index and run a real query against it

> **Data source:** this notebook uses `data/WHO_Hypertension_Guideline_2021.pdf`, the real
> WHO guideline bundled with your starter kit — not a toy example.


## 0. Setup

Run this cell first. It adds the repo root to the path so we can reuse the exact same
functions defined in `ingest.py`, and confirms your environment is ready.


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from pathlib import Path

print("Data directory:", config.DATA_DIR)
print("Chunk size (tokens):", config.CHUNK_SIZE)
print("Chunk overlap (tokens):", config.CHUNK_OVERLAP)
print("PDFs found:", [p.name for p in config.DATA_DIR.glob("*.pdf")])


## 1. Why Grounding Matters

Before touching any code, sit with this for a second: a large language model can generate
a fluent, confident-sounding clinical recommendation **even when it has no real evidence
behind it.** It has no built-in mechanism to say "I don't know."

Retrieval-Augmented Generation (RAG) fixes this by separating two things:

- **What the model knows** (its training data — broad, but unverifiable and possibly stale)
- **What the model is allowed to say** (only what's in the text you hand it right now)

Everything you build today is the **first half** of that separation: turning a trustworthy
PDF into a searchable, citable index. Day 3 builds the second half (forcing the model to
answer only from what this index returns).


## 2. Step 1 — Parse the PDF

`PyPDFLoader` reads a PDF and returns one LangChain `Document` per page, each carrying
page-level metadata automatically (page number, source path).

Run the cell below and inspect the output. Notice that `page.metadata["page"]` is
**zero-indexed** — page index `0` is the PDF's first page.


In [1]:
import os
import sys 
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
sys.path.append(str(Path.cwd() / "CDSS"))
from src import config
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

pdf_path = list(config.RAW_DATA_DIR.glob("*.pdf"))[0]
print(f"Loading: {pdf_path.name}\n")

loader = PyMuPDFLoader(str(pdf_path))
raw_pages = loader.load()

print(f"Loaded {len(raw_pages)} pages.\n")
print("--- Page 3 (index 2) raw metadata, as PyPDFLoader gives it to us ---")
print(raw_pages[2].metadata)
print("\n--- Page 3 (index 2) first 400 characters ---")
print(raw_pages[2].page_content[:400])


C:\Users\ABDELRAHMAN GAMAL\AppData\Local\Temp\ipykernel_18148\1203951809.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Loading: chapter 5 (1).pdf

Loaded 10 pages.

--- Page 3 (index 2) raw metadata, as PyPDFLoader gives it to us ---
{'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\chapter 5 (1).pdf', 'file_path': 'C:\\Users\\ABDELRAHMAN GAMAL\\Desktop\\CDSS\\data\\raw\\chapter 5 (1).pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-17T11:34:55+00:00', 'trapped': '', 'modDate': 'D:20260817113455Z', 'creationDate': '', 'page': 2}

--- Page 3 (index 2) first 400 characters ---
5 Planning primary treatment 
Staging lung cancer 
The T scores are based on: 
Lung cancer is staged using the tumor, node, 
> The biggest width of the primary tumor 
metastasis (TNM) system maintained by the 
measured in centimeters (cm) 
American Joint Committee on Cancer. Each 
letter in the system represents an area of 
cancer growth and is assigned a score. 
> Invasive growth o

### Raw metadata isn't citation-ready yet

Look at the metadata above: PyPDFLoader gives you a generic `page` index and a full file
`source` path — but nothing called `document_name`, and no human-friendly 1-indexed page
number. If we chunk these pages as-is, every citation later would say "unknown, page ?".

`load_pdfs()` in `ingest.py` does one small but critical thing: it stamps
`document_name` and a 1-indexed `page_number` onto every page's metadata **before**
chunking, so that metadata survives all the way through to the final citation.


In [2]:
data_dir = Path("data/raw") 
all_pages = []
# Use a reliable PDF loader to parse the documents
for pdf_path in data_dir.glob("*.pdf"):
    loader = PyMuPDFLoader(str(pdf_path))
    pages = loader.load()
    
    for page in pages:
        page.metadata['document_name'] = pdf_path.name
        current_page = page.metadata.get('page', 0)
        page.metadata['page_number'] = current_page + 1 if isinstance(current_page, int) else 1
        
    all_pages.extend(pages)
print(f"Successfully loaded {len(all_pages)} pages with normalized metadata.")

Successfully loaded 31 pages with normalized metadata.


### Checkpoint 1

Look at the printed text above. Answer for yourself before moving on:

- Are section headings ("3.1 Blood pressure threshold...") visible as recognizable text, or
  did they get mangled?
- Are there any obvious parsing artifacts (broken words, merged columns, stray characters)?

If parsing looks clean here, section-aware chunking (Step 2) will work well. If it looks
messy, no chunking strategy will fully save you — the fix belongs upstream, in parsing.


## 3. Step 2 — Compare Chunking Strategies

We'll build **two** chunkers on the same pages and compare them directly: a naive
fixed-size splitter, and the section-aware splitter actually used in `ingest.py`.


In [3]:

naive_splitter = CharacterTextSplitter(
    separator="", 
    chunk_size=1000, 
    chunk_overlap=200
)
naive_chunks = naive_splitter.split_documents(all_pages)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
recursive_chunks = recursive_splitter.split_documents(all_pages)

print(f"Naive Chunks created: {len(naive_chunks)}")
print(f"Recursive Chunks created: {len(recursive_chunks)}")

if naive_chunks and recursive_chunks:
    print("\n--- Naive Chunk 1 (May break words/sentences abruptly) ---")
    print(repr(naive_chunks[0].page_content[:200] + "..."))
    
    print("\n--- Recursive Chunk 1 (Preserves semantic boundaries) ---")
    print(repr(recursive_chunks[0].page_content[:200] + "..."))


Naive Chunks created: 99
Recursive Chunks created: 101

--- Naive Chunk 1 (May break words/sentences abruptly) ---
'5 \nPlanning primary treatment \nTypes of primary treatment \nStaging lung cancer \nLung cancer stages \nTreatment by clinical stage \nParticipating in a clinical trial \nKey points \nQuestions to ask \nNCCN G...'

--- Recursive Chunk 1 (Preserves semantic boundaries) ---
'5 \nPlanning primary treatment \nTypes of primary treatment \nStaging lung cancer \nLung cancer stages \nTreatment by clinical stage \nParticipating in a clinical trial \nKey points \nQuestions to ask \nNCCN G...'


In [ ]:
# Look at one naive chunk boundary — notice it can cut mid-sentence
print("--- Naive chunk #5 (often cuts mid-sentence) ---")
print(repr(naive_chunks[5].page_content))

print("\n--- Section-aware chunk #5 (respects paragraph breaks) ---")
print(repr(aware_chunks[5].page_content[:300]))


### Checkpoint 2

Compare the two printed chunks above.

- Does the naive chunk end mid-word or mid-sentence?
- Does the section-aware chunk end at a more natural paragraph or sentence boundary?

This is the entire argument for section-aware chunking in one comparison: **the boundary
you cut at becomes the boundary a citation has to point to.** A citation that lands mid-sentence
is much harder for a clinician to trust and verify.


## 4. Step 3 — Attach Citation Metadata

A chunk without a traceable source is useless for a clinical tool. Before embedding
anything, every chunk needs: **document name, page number, and a stable chunk id.**
This is exactly what `chunk_documents()` in `ingest.py` does — let's call it directly.


In [4]:

embedding_model = HuggingFaceEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")
# Sample clinical statements
statement_1 = "The patient has a history of cardiovascular disease and hypertension."
statement_2 = "High blood pressure and heart disease are present in the patient's medical history."
statement_3 = "The patient presented with a fractured tibia after a skiing accident."
# Generate vector embeddings
embeddings = embedding_model.embed_documents([statement_1, statement_2, statement_3])
# Calculate and print cosine similarity
# Compare statement 1 & 2 (Related Concepts)
sim_related = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
# Compare statement 1 & 3 (Unrelated Concepts)
sim_unrelated = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]
print(f"Cosine Similarity (Related Medical Concepts): {sim_related:.4f}")
print(f"Cosine Similarity (Unrelated Concepts):       {sim_unrelated:.4f}")
print("\nConclusion: Related medical concepts score significantly higher than unrelated ones.")



C:\Users\ABDELRAHMAN GAMAL\AppData\Local\Temp\ipykernel_18148\3660036990.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Cosine Similarity (Related Medical Concepts): 0.7904
Cosine Similarity (Unrelated Concepts):       0.2644

Conclusion: Related medical concepts score significantly higher than unrelated ones.


## 5. Step 4 — What Is an Embedding, Really?

An embedding model converts text into a list of numbers (a **vector**) that captures
meaning — texts about similar topics end up as vectors that point in similar directions,
even if they don't share any of the same words.

Let's embed three short phrases and check which two are "closer" in vector space.


In [5]:
# Build and persist a local Chroma vector database
vector_store_dir = "vector_store"

# We use the recursive_chunks as they preserve semantic boundaries
vector_store = Chroma.from_documents(
    documents=recursive_chunks,
    embedding=embedding_model,
    persist_directory=vector_store_dir
)

print(f"Vector store successfully built and persisted to '{vector_store_dir}'.")

# Execute a final clinical query
query = "What is the target blood pressure for a patient with cardiovascular disease?"
print(f"\nExecuting Query: '{query}'")

# Perform similarity search with relevance scores (k=3)
results = vector_store.similarity_search_with_relevance_scores(query, k=3)

print("\n--- Top Relevant Passages ---")
for i, (doc, score) in enumerate(results, 1):
    doc_name = doc.metadata.get('document_name', 'Unknown')
    page_num = doc.metadata.get('page_number', 'Unknown')
    
    print(f"\nResult {i} (Relevance Score: {score:.4f})")
    print(f"Document Name: {doc_name}")
    print(f"Page Number: {page_num}")
    print(f"Passage:\n{doc.page_content[:300]}...\n")


Vector store successfully built and persisted to 'vector_store'.

Executing Query: 'What is the target blood pressure for a patient with cardiovascular disease?'

--- Top Relevant Passages ---

Result 1 (Relevance Score: -190.4630)
Document Name: lung cancer (1).pdf
Page Number: 16
Passage:
PLEASE NOTE that use of this NCCN Content is governed by the End-User License Agreement, and you MAY NOT distribute this Content or use it with any artficialinteligence model or tool. 
Comprehensive NCCN Guidelines Version 1.2027 
% 
ey o, 
Small Cell Lung Cancer 
dablegronents 
PRINCIPLES OF SUPPOR...


Result 2 (Relevance Score: -191.2241)
Document Name: chapter 5 (1).pdf
Page Number: 7
Passage:
against a standard treatment. If the 
results are good, it may be approved by 
the FDA. 
> Phase 4 trials study the safety and 
benefit of an FDA-approved treatment. 
Who can enroll? 
It depends on the clinical trial’s rules, called 
eligibility criteria. The rules may be about age, 
cancer type and...


R

C:\Users\ABDELRAHMAN GAMAL\AppData\Local\Temp\ipykernel_18148\2364744564.py:18: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'format': 'PDF 1.5', 'file_path': 'data\\raw\\lung cancer (1).pdf', 'keywords': '', 'trapped': '', 'modDate': 'D:20260817113141Z', 'creationdate': '', 'title': '', 'page': 15, 'producer': 'iLovePDF', 'subject': '', 'source': 'data\\raw\\lung cancer (1).pdf', 'total_pages': 21, 'page_number': 16, 'document_name': 'lung cancer (1).pdf', 'moddate': '2026-08-17T11:31:41+00:00', 'author': '', 'creator': '', 'creationDate': ''}, page_content='PLEASE NOTE that use of this NCCN Content is governed by the End-User License Agreement, and you MAY NOT distribute this Content or use it with any artficialinteligence model or tool. \nComprehensive NCCN Guidelines Version 1.2027 \n% \ney o, \nSmall Cell Lung Cancer \ndablegronents \nPRINCIPLES OF SUPPORTIVE CARE \n+ Smoking cessation advice, counseling, and pharmacotherapy \n» Use the 5 A’s Fra

### Checkpoint 3

You should see the first similarity score noticeably **higher** than the second — the
"same meaning, different words" pair should score closer together, even though they share
almost no words in common. This is the entire mechanism semantic search relies on. If both
scores came out similar, something about the embedding model would be worth investigating
before trusting it on Day 2.


## 6. Step 5 — Build the Vector Index

Now we embed every chunk and store it in a local ChromaDB collection, using the exact
same `build_index()` function from `ingest.py`. This is the same call the script makes —
seeing it run here just makes the process visible.

> First run downloads a small local embedding model (~100MB) — this happens once and is
> cached afterward.


In [ ]:
from ingest import build_index

vectordb = build_index(chunks)
print("\nIndex build complete.")


## 7. Step 6 — Run a Real Query

The whole point of everything above: ask a real clinical question and see whether the
index returns something relevant.


In [ ]:
question = "What is the target blood pressure for a patient with cardiovascular disease?"

results = vectordb.similarity_search_with_relevance_scores(question, k=3)

print(f"Question: {question}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"[{i}] score={score:.3f}  {doc.metadata.get('document_name')}, "
          f"page {doc.metadata.get('page_number')}")
    print(f"    \"{doc.page_content[:180].strip()}...\"\n")


### Checkpoint 4 — Day 1 Self-Check

Before you close this notebook, confirm all of the following are true:

- [ ] The top retrieved chunk in Step 6 is genuinely relevant to the question asked
- [ ] Every result shows a document name and page number (not `None`)
- [ ] You could explain to a teammate, in one sentence, why section-aware chunking beat
      the naive splitter in Checkpoint 2

If any of these aren't true yet, that's normal — go back to the relevant step above and
adjust `config.py` (chunk size, overlap) before moving on to Day 2.

## What's Next

Day 2's notebook picks up exactly here: tuning `top_k`, benchmarking this embedding model
against alternatives, and proving your retrieval quality with real, logged numbers instead
of a single example query.
